# SP500 Executive Role Classification - Fine-Tuning GPT-OSS-20B

Fine-tunes OpenAI's gpt-oss-20b (21B total / 3.6B active MoE) to classify executive ranks and roles from SP500 proxy filings.

**GPU**: Works on free Colab T4 (16GB) — QLoRA needs ~14GB VRAM.

**Runtime**: Go to Runtime → Change runtime type → Select **T4 GPU** (or A100 for faster training).

## 0. Configuration

Set all key parameters here. Change these before running the rest of the notebook.

In [ ]:
# ============================================================
# CONFIGURATION - Optimized for A100 80GB
# ============================================================

# GitHub repo
GITHUB_REPO = "https://github.com/daalbert981/finetune_sp500.git"

# Base model — OpenAI gpt-oss-20b (21B total, 3.6B active MoE)
MODEL_NAME = "unsloth/gpt-oss-20b"

# HuggingFace Hub settings
HF_REPO_ID = "daresearch/sp500-exec-classifier"
HF_TOKEN = ""  # paste your HF write token (from https://huggingface.co/settings/tokens)

# Training hyperparameters — A100 80GB allows larger batches and sequences
TRAIN_VALID_SPLIT = 0.95       # 95% train, 5% validation
NUM_EPOCHS = 3                 # 3-5 for this dataset size
LEARNING_RATE = 2e-4           # standard for QLoRA fine-tuning
BATCH_SIZE = 8                 # A100 80GB can handle 8 easily
GRADIENT_ACCUMULATION = 2      # effective batch = 8 * 2 = 16
MAX_SEQ_LENGTH = 2048          # A100 has headroom for longer context
WARMUP_RATIO = 0.03
WEIGHT_DECAY = 0.01
SEED = 42

# LoRA hyperparameters — higher rank affordable on A100
LORA_R = 128                   # more capacity with 80GB headroom
LORA_ALPHA = 32                # scale with rank
LORA_DROPOUT = 0.0

# Quantization
LOAD_IN_4BIT = True            # still use 4-bit to leave room for large batches

## 1. Install Dependencies

In [ ]:
%%capture
# Install Unsloth from source (matches official gpt-oss notebook exactly)
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch>=2.8.0" "triton>=3.4.0" {_numpy} {_pil} torchvision bitsandbytes "transformers==4.56.2" \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
        git+https://github.com/triton-lang/triton.git@0add68262ab0a2e33b84524346cb27cbb2787356#subdirectory=python/triton_kernels
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
# Force correct versions AFTER install (critical — prevents transformers 5.x override)
!uv pip install --upgrade --no-deps transformers==4.56.2 tokenizers trl==0.22.2 unsloth unsloth_zoo

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    raise RuntimeError("No GPU detected! Go to Runtime → Change runtime type → GPU")

## 2. Clone Repo & Load Data

In [ ]:
import os

REPO_DIR = "/content/finetune_sp500"
if not os.path.exists(REPO_DIR):
    !git clone {GITHUB_REPO} {REPO_DIR}
else:
    print(f"Repo already cloned at {REPO_DIR}")

DATA_PATH = os.path.join(REPO_DIR, "Training Datasets", "full_data_n_4977.jsonl")
assert os.path.exists(DATA_PATH), f"Data file not found at {DATA_PATH}"

In [ ]:
import json

# Load the clean JSONL (already in chat message format, all 4,977 consistent)
with open(DATA_PATH, "r") as f:
    examples = [json.loads(line) for line in f]

print(f"Total examples: {len(examples)}")

# Verify consistency
sample = examples[0]
print(f"\nMessage structure: {[m['role'] for m in sample['messages']]}")
print(f"System prompt length: {len(sample['messages'][0]['content'])} chars")
print(f"\n--- Sample ---")
print(f"User:      {sample['messages'][1]['content'][:150]}")
print(f"Assistant: {sample['messages'][2]['content']}")

# Sanity check: all have both <rank> and <role>
assert all("<rank>" in e["messages"][2]["content"] and "<role>" in e["messages"][2]["content"] for e in examples), \
    "Not all examples have both <rank> and <role> tags!"
print(f"\nAll {len(examples)} examples have both <rank> and <role> tags.")

## 3. Format Data for Chat Fine-Tuning & Train/Valid Split

In [ ]:
from datasets import Dataset
import random

random.seed(SEED)

# Shuffle and split
random.shuffle(examples)
split_idx = int(len(examples) * TRAIN_VALID_SPLIT)

train_dataset = Dataset.from_list(examples[:split_idx])
valid_dataset = Dataset.from_list(examples[split_idx:])

print(f"Train: {len(train_dataset)} examples")
print(f"Valid: {len(valid_dataset)} examples")

## 4. Load Model with Unsloth

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
    dtype=None,  # auto-detect
    full_finetuning=False,
)

print(f"Model loaded: {MODEL_NAME}")
print(f"Model parameters: {model.num_parameters() / 1e9:.1f}B")

## 5. Configure LoRA Adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias="none",
    use_gradient_checkpointing="unsloth",  # Unsloth optimized — saves VRAM
    random_state=SEED,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable / 1e6:.1f}M / {total / 1e9:.1f}B ({100 * trainable / total:.2f}%)")
print(f"GPU VRAM used: {torch.cuda.max_memory_reserved() / 1e9:.2f} GB")

## 5b. Apply Chat Template & Format Dataset

In [ ]:
from unsloth.chat_templates import get_chat_template, standardize_sharegpt

# Apply gpt-oss chat template to tokenizer (must happen after model loading)
tokenizer = get_chat_template(tokenizer, chat_template="gpt-oss")

# Formatting function: apply chat template to create "text" field
def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return {"text": texts}

# Standardize and pre-format datasets
train_dataset = standardize_sharegpt(train_dataset)
train_dataset = train_dataset.map(formatting_prompts_func, batched=True)

valid_dataset = standardize_sharegpt(valid_dataset)
valid_dataset = valid_dataset.map(formatting_prompts_func, batched=True)

print(f"Sample formatted text (first 300 chars):")
print(train_dataset[0]["text"][:300])

## 6. Training

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    args=SFTConfig(
        output_dir="./outputs",
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        warmup_ratio=WARMUP_RATIO,
        lr_scheduler_type="cosine",
        bf16=True,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=50,
        save_strategy="steps",
        save_steps=100,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        seed=SEED,
        report_to="none",
        max_seq_length=MAX_SEQ_LENGTH,
        dataloader_num_workers=4,
        dataloader_pin_memory=True,
    ),
)

print(f"Effective batch size: {BATCH_SIZE * GRADIENT_ACCUMULATION}")
print(f"Total training steps: ~{len(train_dataset) * NUM_EPOCHS // (BATCH_SIZE * GRADIENT_ACCUMULATION)}")

In [ ]:
# GPU memory before training
gpu_stats = torch.cuda.get_device_properties(0)
reserved = torch.cuda.max_memory_reserved() / 1e9
print(f"GPU memory reserved before training: {reserved:.2f} GB / {gpu_stats.total_memory / 1e9:.2f} GB")

In [ ]:
# Train!
train_result = trainer.train()

print(f"\nTraining complete!")
print(f"Train loss: {train_result.training_loss:.4f}")
print(f"Train runtime: {train_result.metrics['train_runtime'] / 60:.1f} min")
print(f"Peak GPU memory: {torch.cuda.max_memory_reserved() / 1e9:.2f} GB")

## 7. Quick Validation Check

In [ ]:
# Run evaluation on validation set
eval_results = trainer.evaluate()
print(f"Validation loss: {eval_results['eval_loss']:.4f}")
print(f"Validation perplexity: {2.71828 ** eval_results['eval_loss']:.2f}")

In [ ]:
# Test inference on a few validation examples
FastLanguageModel.for_inference(model)

num_test = 5
correct = 0

for i in range(min(num_test, len(valid_dataset))):
    messages = valid_dataset[i]["messages"]
    prompt_messages = messages[:2]  # system + user only

    inputs = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs,
            max_new_tokens=200,
            temperature=0.0,
            do_sample=False,
        )

    generated = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True).strip()
    expected = messages[2]["content"]

    match = generated.strip() == expected.strip()
    correct += int(match)

    print(f"\n--- Example {i+1} {'✓' if match else '✗'} ---")
    print(f"Input:    {messages[1]['content'][:120]}")
    print(f"Expected: {expected}")
    print(f"Got:      {generated}")

print(f"\nExact match: {correct}/{num_test}")

## 8. Save to HuggingFace Hub

Three save options:
1. **LoRA adapters only** (small, ~100-300MB) — loads on top of base model
2. **Merged 16-bit** (full model, largest) — standalone, no base model needed
3. **Merged 4-bit GGUF** (for llama.cpp / Ollama deployment)

In [ ]:
# Login to HuggingFace
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)
else:
    print("No HF_TOKEN set. Run: huggingface-cli login")
    !huggingface-cli login

In [ ]:
# Option A: Save LoRA adapters only (recommended for flexibility)
model.save_pretrained_merged(
    "./outputs/merged_model",
    tokenizer,
    save_method="merged_16bit",
)
print("Merged model saved locally.")

In [ ]:
# Push LoRA adapters to HuggingFace Hub
model.push_to_hub_merged(
    HF_REPO_ID,
    tokenizer,
    save_method="merged_16bit",
    token=HF_TOKEN if HF_TOKEN else None,
)
print(f"Model pushed to https://huggingface.co/{HF_REPO_ID}")

In [ ]:
# Option B: Also save GGUF for llama.cpp / Ollama (optional)
# Uncomment to save in GGUF format:

# model.push_to_hub_gguf(
#     HF_REPO_ID + "-gguf",
#     tokenizer,
#     quantization_method="q4_k_m",
#     token=HF_TOKEN if HF_TOKEN else None,
# )
# print(f"GGUF model pushed to https://huggingface.co/{HF_REPO_ID}-gguf")

## 9. Holdout Evaluation

Runs the fine-tuned model on all 2,010 holdout examples, parses predictions, and computes per-label accuracy, precision, recall, and F1. Also exports results to CSV for further analysis.

In [ ]:
import csv
import re
import time

FULL_SYSTEM_PROMPT = """This assistant is trained to code executive ranks and roles along the following categories with 1 or 0.

Ranks:
- VP: 1 if Vice President (VP), 0 otherwise
- SVP: 1 if Senior Vice President (SVP), 0 otherwise
- EVP: 1 if Executive Vice President (EVP), 0 otherwise
- SEVP: 1 if Senior Executive Vice President (SEVP), 0 otherwise
- Director: 1 if Director, 0 otherwise
- Senior Director: 1 if Senior Director, 0 otherwise
- MD: 1 if Managing Director (MD), 0 otherwise
- SMD: 1 if Senior Managing Director (SMD), 0 otherwise
- SE: 1 if Senior Executive, 0 otherwise
- VC: 1 if Vice Chair (VC), 0 otherwise
- SVC: 1 if Senior Vice Chair (SVC), 0 otherwise
- President: 1 if President of the parent company, 0 when President of subsidiary or division but not parent company.

Roles:
- Board: 1 when role suggests person is a member of the board of directors, 0 otherwise
- CEO: 1 when Chief Executive Officer of parent company, 0 when Chief Executive Officer of a subsidiary but not parent company.
- CXO: 1 when C-Suite title, i.e., Chief X Officer, where X can be any type of designation, 0 otherwise. Chief Executive Officer of the parent company. Not Chief AND Officer, e.g., only officer of a function.
- Primary: 1 when responsible for primary activity of value chain, i.e., Supply Chain, Manufacturing, Operations, Marketing & Sales, Customer Service and alike, 0 when not a primary value chain activity.
- Support: 1 when responsible for a support activity of the value chain, i.e., Procurement, IT, HR, Management, Strategy, HR, Finance, Legal, R&D, Investor Relations, Technology, General Counsel and alike, 0 when not support activity of the value.
- BU: 1 when involved with an entity/distinct unit responsible for Product, Customer, or Geographical domain/unit; or role is about a subsidiary, 0 when responsibility is not for a specific product/customer/geography area but, for example, for the entire parent company."""

# All label columns in the holdout CSV
RANK_LABELS = ["vp", "svp", "evp", "sevp", "dir", "sdir", "md", "smd", "se", "vc", "svc", "president"]
ROLE_LABELS = ["board", "ceo", "cxo", "primary", "support", "bu"]
ALL_LABELS = RANK_LABELS + ROLE_LABELS


def predict_single(role_title: str, company: str, year: int, name: str) -> str:
    """Run inference on a single executive."""
    user_msg = (
        f"In {year} the company '{company}' had an executive with the name {name}, "
        f"whose official role title was: '{role_title}'."
    )
    messages = [
        {"role": "system", "content": FULL_SYSTEM_PROMPT},
        {"role": "user", "content": user_msg},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs, max_new_tokens=200, temperature=0.0, do_sample=False
        )
    return tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True).strip()


def parse_output(text: str) -> dict:
    """Parse <rank>vp=0;svp=1;...</rank><role>board=0;...</role> into dict."""
    result = {}
    for tag in ["rank", "role"]:
        match = re.search(f"<{tag}>(.*?)</{tag}>", text)
        if match:
            for pair in match.group(1).split(";"):
                if "=" in pair:
                    k, v = pair.split("=", 1)
                    try:
                        result[k.strip()] = int(v.strip())
                    except ValueError:
                        result[k.strip()] = -1  # flag unparseable
    return result

print("Inference helpers loaded.")

In [ ]:
# Load holdout data
HOLDOUT_PATH = os.path.join(REPO_DIR, "Training Datasets", "holdout_labeling_09072024_template.csv")

with open(HOLDOUT_PATH, "r", encoding="latin-1") as f:
    reader = csv.DictReader(f)
    holdout_rows = list(reader)

print(f"Holdout examples: {len(holdout_rows)}")

# Run predictions on ALL holdout rows (takes ~15-30 min on A100)
FastLanguageModel.for_inference(model)

predictions = []
parse_failures = 0
t0 = time.time()

for i, row in enumerate(holdout_rows):
    raw = predict_single(row["role"], row["company"], int(row["year"]), row["full_name"])
    parsed = parse_output(raw)

    # Check if parse returned all expected labels
    if not all(lbl in parsed for lbl in ALL_LABELS):
        parse_failures += 1

    predictions.append({
        "idx": i,
        "uniqueid": row["uniqueid"],
        "full_name": row["full_name"],
        "company": row["company"],
        "role": row["role"],
        "year": row["year"],
        "raw_output": raw,
        "parsed": parsed,
    })

    if (i + 1) % 100 == 0:
        elapsed = time.time() - t0
        rate = (i + 1) / elapsed
        eta = (len(holdout_rows) - i - 1) / rate
        print(f"  [{i+1}/{len(holdout_rows)}] {rate:.1f} ex/s, ETA {eta/60:.1f} min, parse failures so far: {parse_failures}")

elapsed = time.time() - t0
print(f"\nDone! {len(predictions)} predictions in {elapsed/60:.1f} min")
print(f"Parse failures (missing labels): {parse_failures}/{len(predictions)}")

In [ ]:
import numpy as np

# Build truth and prediction arrays
y_true = {lbl: [] for lbl in ALL_LABELS}
y_pred = {lbl: [] for lbl in ALL_LABELS}

for pred, row in zip(predictions, holdout_rows):
    for lbl in ALL_LABELS:
        truth = int(row[lbl])
        predicted = pred["parsed"].get(lbl, -1)
        y_true[lbl].append(truth)
        y_pred[lbl].append(predicted)

# Per-label metrics
print(f"{'Label':>10} | {'Acc':>6} | {'Prec':>6} | {'Recall':>6} | {'F1':>6} | {'Support':>7} | {'Parse%':>6}")
print("-" * 72)

label_metrics = {}
for lbl in ALL_LABELS:
    yt = np.array(y_true[lbl])
    yp = np.array(y_pred[lbl])

    # Only evaluate where parse succeeded (predicted is 0 or 1)
    valid = (yp >= 0)
    parse_rate = valid.sum() / len(yp) * 100

    yt_v = yt[valid]
    yp_v = yp[valid]

    if len(yt_v) == 0:
        label_metrics[lbl] = {"acc": 0, "prec": 0, "rec": 0, "f1": 0, "support": 0, "parse_rate": 0}
        print(f"{lbl:>10} | {'N/A':>6} | {'N/A':>6} | {'N/A':>6} | {'N/A':>6} | {0:>7} | {parse_rate:>5.1f}%")
        continue

    tp = ((yp_v == 1) & (yt_v == 1)).sum()
    fp = ((yp_v == 1) & (yt_v == 0)).sum()
    fn = ((yp_v == 0) & (yt_v == 1)).sum()
    tn = ((yp_v == 0) & (yt_v == 0)).sum()

    acc = (tp + tn) / len(yt_v)
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    support = int(yt_v.sum())

    label_metrics[lbl] = {"acc": acc, "prec": prec, "rec": rec, "f1": f1, "support": support, "parse_rate": parse_rate}
    print(f"{lbl:>10} | {acc:>5.1%} | {prec:>5.1%} | {rec:>5.1%} | {f1:>5.1%} | {support:>7} | {parse_rate:>5.1f}%")

# Summary
print("-" * 72)
avg_acc = np.mean([m["acc"] for m in label_metrics.values()])
avg_f1 = np.mean([m["f1"] for m in label_metrics.values()])
avg_parse = np.mean([m["parse_rate"] for m in label_metrics.values()])
print(f"{'MACRO AVG':>10} | {avg_acc:>5.1%} |        |        | {avg_f1:>5.1%} |         | {avg_parse:>5.1f}%")

# Group summaries
rank_f1 = np.mean([label_metrics[l]["f1"] for l in RANK_LABELS])
role_f1 = np.mean([label_metrics[l]["f1"] for l in ROLE_LABELS])
print(f"\nRank labels macro F1: {rank_f1:.1%}")
print(f"Role labels macro F1: {role_f1:.1%}")

In [ ]:
# Show misclassified examples for inspection
print("=== MISCLASSIFIED EXAMPLES (first 20) ===\n")
misclassified = []

for pred, row in zip(predictions, holdout_rows):
    errors = {}
    for lbl in ALL_LABELS:
        truth = int(row[lbl])
        predicted = pred["parsed"].get(lbl, -1)
        if predicted != truth:
            errors[lbl] = f"pred={predicted} truth={truth}"
    if errors:
        misclassified.append((pred, row, errors))

print(f"Total rows with at least 1 error: {len(misclassified)} / {len(holdout_rows)} ({len(misclassified)/len(holdout_rows):.1%})")
print(f"Fully correct rows: {len(holdout_rows) - len(misclassified)} / {len(holdout_rows)} ({(len(holdout_rows)-len(misclassified))/len(holdout_rows):.1%})\n")

for pred, row, errors in misclassified[:20]:
    print(f"{row['full_name']} — {row['role']} @ {row['company']} ({row['year']})")
    for lbl, err in errors.items():
        print(f"    {lbl}: {err}")
    print()

In [ ]:
# Export full results to CSV for offline analysis
OUTPUT_CSV = "/content/holdout_results.csv"

with open(OUTPUT_CSV, "w", newline="") as f:
    fieldnames = ["uniqueid", "year", "company", "full_name", "role", "raw_output"] + \
                 [f"truth_{l}" for l in ALL_LABELS] + \
                 [f"pred_{l}" for l in ALL_LABELS] + \
                 [f"correct_{l}" for l in ALL_LABELS]
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()

    for pred, row in zip(predictions, holdout_rows):
        out = {
            "uniqueid": row["uniqueid"],
            "year": row["year"],
            "company": row["company"],
            "full_name": row["full_name"],
            "role": row["role"],
            "raw_output": pred["raw_output"],
        }
        for lbl in ALL_LABELS:
            truth = int(row[lbl])
            predicted = pred["parsed"].get(lbl, -1)
            out[f"truth_{lbl}"] = truth
            out[f"pred_{lbl}"] = predicted
            out[f"correct_{lbl}"] = int(predicted == truth)
        writer.writerow(out)

print(f"Results exported to {OUTPUT_CSV}")
print("Download: from google.colab import files; files.download('{OUTPUT_CSV}')")

# Auto-download in Colab
try:
    from google.colab import files
    files.download(OUTPUT_CSV)
except ImportError:
    pass